In [ ]:
import os
import re
import csv

base_dir = "/project_cephfs/3022017.06/UKB/freesurfer/1875529/scripts/recon-all.log"
subject = "1875529"
output_csv = "euler_numbers.csv"

pattern = re.compile(r"orig\.nofix lheno =\s*([\-0-9]+), rheno =\s*([\-0-9]+)")

rows = []


#recon_all_path = os.path.join(base_dir, subject, "scripts", "recon-all.log")
recon_all_path = base_dir

if not os.path.isfile(recon_all_path):
    print(f"Missing log for {subject}")


lheno, rheno = None, None

with open(recon_all_path, "r") as f:
    for line in f:
        match = pattern.search(line)
        if match:
            lheno, rheno = match.groups()
            break

rows.append([subject, lheno, rheno])

print(rows)
# with open(output_csv, "w", newline="") as f:
#     writer = csv.writer(f)
#     writer.writerow(["subjectID", "lheno", "rheno"])
#     writer.writerows(rows)


[['1875529', '-24', '-34']]


In [ ]:
import os
import re
import csv

base_dir = "/project_cephfs/3022017.06/UKB/freesurfer/"
output_csv = "euler_numbers.csv"

pattern = re.compile(r"orig\.nofix lheno =\s*([\-0-9]+), rheno =\s*([\-0-9]+)")

rows = []

for subject in os.listdir(base_dir):
    if not subject.isdigit():
        continue

    recon_all_path = os.path.join(base_dir, subject, "scripts", "recon-all.log")

    if not os.path.isfile(recon_all_path):
        print(f"Missing log for {subject}")
        continue

    lh_euler, rh_euler = None, None

    with open(recon_all_path, "r") as f:
        for line in f:
            match = pattern.search(line)
            if match:
                lh_euler, rh_euler = match.groups()
                break

    rows.append([subject, lh_euler, rh_euler])


with open(output_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["SubjectID", "lh_euler", "rh_euler"])
    writer.writerows(rows)


In [ ]:
import pandas as pd
import numpy as np

#euler data
df1 = pd.read_csv("/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/csv_files/euler_numbers.csv")
df1

#data with correct site
df2 = pd.read_csv("/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/final_data/Final_data_corrected_site.csv")

df_extra = df2[["SubjectID", "Site"]]


df3 = df1.merge(df_extra, on="SubjectID", how="right")

df3 = df3[df3["Site"] > 0]
df3 = df3.astype({"SubjectID": int, "lh_euler": int, "rh_euler": int, "Site": int})

df3["avg_euler"] = df3[["lh_euler", "rh_euler"]].mean(axis=1)
site_medians = df3.groupby("Site")["avg_euler"].median()
df3["Site_median"] = df3["Site"].map(site_medians)

df3["avg_euler_centered"] = df3["avg_euler"] - df3["Site_median"]

df3["avg_euler_centered_neg"] = -1 * df3["avg_euler_centered"]
df3["avg_euler_centered_neg_sqrt"] = np.sqrt(np.abs(df3["avg_euler_centered_neg"]))
good_subj_df = df3[df3["avg_euler_centered_neg_sqrt"] < 10]
bad_subj_df = df3[df3["avg_euler_centered_neg_sqrt"] > 10]


In [ ]:
df = pd.read_csv("/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/final_data/euler_nr_correct_site_data.csv")
df.head(80)
